# 10 — Fragility: sensor-model mismatch (claim F3)
S0 (inherited 0.5%-of-value convention) … S4 (no usable concentration measurement); each trained 3 × 40k and
tested on its own sensor model; plus the cross case (trained on S0, tested on S3 data: the mismatch).
Report posterior sd, coverage, Corr(α, β) per scenario (artefact correlations), α information without C.

**PRE-REGISTER:** silent failure = β or α coverage < 0.5 with leakage < 1% and sd within 20% of the matched
posterior's.

In [ ]:
import json, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
NB_DIR = Path.cwd().resolve()
assert NB_DIR.name == "notebooks" and NB_DIR.parent.name == "system_III", f"run from system_III/notebooks (cwd {NB_DIR})"
ROOT = NB_DIR.parent
sys.path.insert(0, str(ROOT / "src"))
import importlib, system_iii
from system_iii import physics as ph, simulator as sim, features as fe, data as da, scenarios as scn
from system_iii import sensors as sen, information as info, config as cfg, rules, metrics as wm
for _m in (ph, sim, fe, da, scn, sen, info, cfg, rules, wm):
    importlib.reload(_m)
R, FIGS, DATA = ROOT / "results", ROOT / "figures", ROOT / "data"
for _d in (R, FIGS, DATA):
    _d.mkdir(exist_ok=True)
P = ph.PLANT
T0 = time.time()

In [ ]:
from system_iii import inference as inf
import pickle
rows = []; posts = {}
for s_model in sen.SENSORS:
    f = DATA / f"10_bank_{s_model}.npz"
    if not f.exists():
        da.save_npz(f, da.make_windows(sim.prior_draws(120_000, 10100), 10100, sensors=s_model))
    b = da.load_npz(f); ms = []
    for s in range(3):
        g = R / "10" / f"post_{s_model}_s{s}.pkl"; g.parent.mkdir(exist_ok=True)
        if g.exists():
            ms.append(pickle.load(open(g, "rb")))
        else:
            sl = slice(s * 40_000, (s + 1) * 40_000)
            p_, _ = inf.train_npe(b["theta"][sl], b["x"][sl], seed=s); pickle.dump(p_, open(g, "wb")); ms.append(p_)
    posts[s_model] = inf.EnsemblePosterior(ms)
for s_model in sen.SENSORS:
    test = da.test_set(10199, sensors=s_model)
    row, sc = inf.evaluate(posts[s_model], test, seed=0); rows += [{"trained": s_model, "tested": s_model, **r} for r in sc]
test3 = da.test_set(10199, sensors="S3")
row, sc = inf.evaluate(posts["S0"], test3, seed=0); rows += [{"trained": "S0", "tested": "S3", **r} for r in sc]
res = pd.DataFrame(rows); res.to_csv(R / "10_sensor_ladder.csv", index=False)
display(res.pivot_table(index="scenario", columns=["trained", "tested"], values=["sd_a", "sd_b", "cov90_truth_b", "corr_ab_mean"]).round(3))

## Interpretation — to be written after the run
Write toward the storyline (`docs/system_III_plan.md` §1) and update the claims ledger (§2).